# Pandas from First Principles
## Notebook 8: Merging & Joining

---

| | |
|---|---|
| **Series** | Pandas from First Principles |
| **Notebook** | 8 of N |
| **Difficulty** | Beginner → Intermediate |
| **Prerequisites** | Notebooks 1–7 (DataFrames, Selection, Filtering, Apply) |

---

### What you will learn

- Why merging DataFrames is a core data skill
- `pd.merge()` — the main Swiss-army-knife tool
- Four join types: **inner, left, right, outer**
- Merging on multiple columns
- Common bugs and how to spot them
- `df.join()` — the index-based shortcut

> **Analogy:** Imagine you have two Excel sheets — one with employee names and another with their salaries — and you need to combine them using the common column `emp_id`. That's exactly what merging does.

---
## 0. Setup — Datasets Used Throughout This Notebook

We will use three small, realistic datasets throughout. Run this cell first.

In [3]:
import pandas as pd

# Dataset 1: Employees
employees = pd.DataFrame({
    'emp_id':  [1, 2, 3, 4, 5],
    'name':    ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept_id': [10, 20, 10, 30, 20],
})

# Dataset 2: Departments
departments = pd.DataFrame({
    'dept_id':   [10, 20, 30, 40],
    'dept_name': ['Engineering', 'Marketing', 'HR', 'Finance'],
    'location':  ['Delhi', 'Mumbai', 'Pune', 'Bangalore'],
})

# Dataset 3: Salaries
# Note: emp_id 4 (Dave) is missing; emp_id 6 is not in employees at all
salaries = pd.DataFrame({
    'emp_id': [1, 2, 3, 5, 6],
    'salary': [85000, 72000, 91000, 75000, 68000],
})

print("=== employees ===")
display(employees)
print()
print("=== departments ===")
display(departments)
print()
print("=== salaries ===")
display(salaries)

=== employees ===


,emp_id,name,dept_id
0,1,Alice,10
1,2,Bob,20
2,3,Carol,10
3,4,Dave,30
4,5,Eve,20



=== departments ===


,dept_id,dept_name,location
0,10,Engineering,Delhi
1,20,Marketing,Mumbai
2,30,HR,Pune
3,40,Finance,Bangalore



=== salaries ===


,emp_id,salary
0,1,85000
1,2,72000
2,3,91000
3,5,75000
4,6,68000


---
## 1. Why Do We Need Merging?

### The Real-World Problem

Data rarely lives in one table. In the real world:

- HR stores **employee info** in one sheet
- Finance stores **salaries** in another sheet
- Operations stores **department details** in yet another

To answer questions like *"What is Alice's salary?"* or *"Which department is in Delhi?"*, you must **combine** these tables using a shared column — called a **key**.

### SQL JOIN Analogy

If you have a SQL background, `pd.merge()` is essentially a SQL `JOIN`:

```sql
-- SQL
SELECT * FROM employees e
INNER JOIN departments d ON e.dept_id = d.dept_id;

-- Pandas equivalent
pd.merge(employees, departments, on='dept_id', how='inner')
```

### The Core Question

> **How do rows from two DataFrames match?**

The answer: by finding rows where the **key column(s)** have the same value.

```
employees       departments
─────────       ───────────
emp_id  dept_id   dept_id  dept_name
  1  →→→→  10   →→→→  10   Engineering  ✓ match!
  2  →→→→  20   →→→→  20   Marketing    ✓ match!
  3  →→→→  10   →→→→  10   Engineering  ✓ match!
  4  →→→→  30   →→→→  30   HR           ✓ match!
  5  →→→→  20   →→→→  20   Marketing    ✓ match!
                →→→→  40   Finance      ✗ no employee!
```

In [8]:
# Let's do our very first merge to see the "combined" view
# This is like asking: "For each employee, what department do they belong to?"

result = pd.merge(employees, departments, on='dept_id')

print("Employees + Department info combined:")
display(result)
print()
print(f"employees shape:   {employees.shape}")
print(f"departments shape: {departments.shape}")
print(f"result shape:      {result.shape}")

Employees + Department info combined:


,emp_id,name,dept_id,dept_name,location
0,1,Alice,10,Engineering,Delhi
1,2,Bob,20,Marketing,Mumbai
2,3,Carol,10,Engineering,Delhi
3,4,Dave,30,HR,Pune
4,5,Eve,20,Marketing,Mumbai



employees shape:   (5, 3)
departments shape: (4, 3)
result shape:      (5, 5)


---
## 2. `pd.merge()` — The Main Tool

### Syntax

```python
pd.merge(
    left,           # left DataFrame
    right,          # right DataFrame
    how='inner',    # join type: 'inner', 'left', 'right', 'outer'
    on=None,        # column name(s) to join on (same name in both)
    left_on=None,   # column in LEFT DataFrame (when names differ)
    right_on=None,  # column in RIGHT DataFrame (when names differ)
    suffixes=('_x', '_y')  # suffixes for overlapping column names
)
```

### Parameters Explained

| Parameter | Type | Purpose | Example |
|-----------|------|---------|--------|
| `left` | DataFrame | First (left) table | `employees` |
| `right` | DataFrame | Second (right) table | `departments` |
| `how` | str | Which rows to keep | `'inner'` (default) |
| `on` | str or list | Key column(s) — **same name in both** | `on='dept_id'` |
| `left_on` | str or list | Key column(s) in left DataFrame | `left_on='emp_id'` |
| `right_on` | str or list | Key column(s) in right DataFrame | `right_on='employee_id'` |
| `suffixes` | tuple | Appended to overlapping column names | `suffixes=('_emp', '_dept')` |

### Return Value

Returns a **new DataFrame** — it does not modify the originals.

### Common Mistakes

- ❌ Using `on=` when the column names differ in the two DataFrames
- ❌ Forgetting to specify `how=` and being surprised by missing rows (default is `'inner'`)
- ❌ Not checking the shape of the result (duplicate rows can silently inflate it)

In [17]:
# Demonstrating: on= vs left_on=/right_on=

# Case 1: Both DataFrames share the column name 'dept_id'
# Use on='dept_id'
merged_on = pd.merge(employees, departments, on='dept_id', how='inner')
print("Case 1 — using on='dept_id':")
display(merged_on[['emp_id', 'name', 'dept_id', 'dept_name']])
print()

# Case 2: Suppose departments had 'department_id' instead of 'dept_id'
departments_renamed = departments.rename(columns={'dept_id': 'department_id'})
print("departments_renamed columns:", list(departments_renamed.columns))

# Now we need left_on and right_on
merged_diff = pd.merge(
    employees,
    departments_renamed,
    left_on='dept_id',       # column in employees
    right_on='department_id' # column in departments_renamed
)
print()
print("Case 2 — using left_on/right_on (different column names):")
display(merged_diff)


Case 1 — using on='dept_id':


,emp_id,name,dept_id,dept_name
0,1,Alice,10,Engineering
1,2,Bob,20,Marketing
2,3,Carol,10,Engineering
3,4,Dave,30,HR
4,5,Eve,20,Marketing



departments_renamed columns: ['department_id', 'dept_name', 'location']

Case 2 — using left_on/right_on (different column names):


,emp_id,name,dept_id,department_id,dept_name,location
0,1,Alice,10,10,Engineering,Delhi
1,2,Bob,20,20,Marketing,Mumbai
2,3,Carol,10,10,Engineering,Delhi
3,4,Dave,30,30,HR,Pune
4,5,Eve,20,20,Marketing,Mumbai


---
## 3. Types of Joins

The `how=` parameter controls **which rows are kept** in the result.

Think of two circles in a Venn diagram — one for the LEFT table, one for the RIGHT table. The overlap is rows that appear in **both** tables.

```
  LEFT           RIGHT
 ┌──────────────────┐
 │      │  overlap  │      │
 │  L   │   L ∩ R   │  R   │
 │ only │           │ only │
 └──────────────────┘

 INNER JOIN  →  only the overlap (L ∩ R)
 LEFT JOIN   →  L only  +  overlap
 RIGHT JOIN  →  overlap +  R only
 OUTER JOIN  →  L only  +  overlap + R only  (everything)
```

| `how=` | Rows kept | NaN appears when... |
|--------|-----------|---------------------|
| `'inner'` | Only rows with a match in **both** | Never |
| `'left'` | **All** left rows; matching right rows | Right columns have no match |
| `'right'` | **All** right rows; matching left rows | Left columns have no match |
| `'outer'` | **All** rows from both | Either side has no match |

### 3a. INNER JOIN — Only Matching Rows (Default)

**Question:** Which employees have a salary record?

```
employees              salaries
emp_id  name           emp_id  salary
  1     Alice    ←→      1     85000   ✓ kept
  2     Bob      ←→      2     72000   ✓ kept
  3     Carol    ←→      3     91000   ✓ kept
  4     Dave             (missing)     ✗ dropped
  5     Eve      ←→      5     75000   ✓ kept
                           6     68000   ✗ dropped (no employee)
```

Result: 4 rows (only the matches)

In [18]:
# INNER JOIN: only rows that exist in BOTH DataFrames
inner_result = pd.merge(employees, salaries, on='emp_id', how='inner')

print("INNER JOIN — employees ∩ salaries:")
display(inner_result)
print()
print("Notice: Dave (emp_id=4) and emp_id=6 are both dropped")
print(f"Result shape: {inner_result.shape}  (expected 4 rows)")

INNER JOIN — employees ∩ salaries:


,emp_id,name,dept_id,salary
0,1,Alice,10,85000
1,2,Bob,20,72000
2,3,Carol,10,91000
3,5,Eve,20,75000



Notice: Dave (emp_id=4) and emp_id=6 are both dropped
Result shape: (4, 4)  (expected 4 rows)


### 3b. LEFT JOIN — All Left Rows, Match Where Possible

**Question:** Show ALL employees; include salary if available, NaN if not.

```
employees              salaries
emp_id  name           emp_id  salary
  1     Alice    ←→      1     85000   ✓ matched → salary = 85000
  2     Bob      ←→      2     72000   ✓ matched → salary = 72000
  3     Carol    ←→      3     91000   ✓ matched → salary = 91000
  4     Dave     ←→      (none)        ✓ KEPT, salary = NaN
  5     Eve      ←→      5     75000   ✓ matched → salary = 75000
                           6     68000   ✗ dropped (not in left)
```

Result: 5 rows (all employees), Dave gets NaN for salary

In [19]:
# LEFT JOIN: keep ALL rows from left (employees), fill NaN for missing right values
left_result = pd.merge(employees, salaries, on='emp_id', how='left')

print("LEFT JOIN — all employees, salary if available:")
display(left_result)
print()
print("Notice: Dave (emp_id=4) is kept with salary = NaN")
print("Notice: emp_id=6 from salaries is dropped (not in left)")
print(f"Result shape: {left_result.shape}  (expected 5 rows = all employees)")

LEFT JOIN — all employees, salary if available:


,emp_id,name,dept_id,salary
0,1,Alice,10,85000.0
1,2,Bob,20,72000.0
2,3,Carol,10,91000.0
3,4,Dave,30,NaN
4,5,Eve,20,75000.0



Notice: Dave (emp_id=4) is kept with salary = NaN
Notice: emp_id=6 from salaries is dropped (not in left)
Result shape: (5, 4)  (expected 5 rows = all employees)


### 3c. RIGHT JOIN — All Right Rows, Match Where Possible

**Question:** Show ALL salary records; include employee info if available, NaN if not.

```
employees              salaries
emp_id  name           emp_id  salary
  1     Alice    ←→      1     85000   ✓ matched
  2     Bob      ←→      2     72000   ✓ matched
  3     Carol    ←→      3     91000   ✓ matched
  4     Dave     ←→      (none)        ✗ dropped (not in right)
  5     Eve      ←→      5     75000   ✓ matched
  (none) NaN     ←→      6     68000   ✓ KEPT, name = NaN
```

Result: 5 rows (all salary records), emp_id=6 gets NaN for name/dept_id

In [20]:
# RIGHT JOIN: keep ALL rows from right (salaries), fill NaN for missing left values
right_result = pd.merge(employees, salaries, on='emp_id', how='right')

print("RIGHT JOIN — all salary records, employee info if available:")
display(right_result)
print()
print("Notice: emp_id=6 is kept with name=NaN, dept_id=NaN")
print("Notice: Dave (emp_id=4) is dropped (not in right/salaries)")
print(f"Result shape: {right_result.shape}  (expected 5 rows = all salary records)")

RIGHT JOIN — all salary records, employee info if available:


,emp_id,name,dept_id,salary
0,1,Alice,10.0,85000
1,2,Bob,20.0,72000
2,3,Carol,10.0,91000
3,5,Eve,20.0,75000
4,6,NaN,NaN,68000



Notice: emp_id=6 is kept with name=NaN, dept_id=NaN
Notice: Dave (emp_id=4) is dropped (not in right/salaries)
Result shape: (5, 4)  (expected 5 rows = all salary records)


### 3d. OUTER JOIN (Full Outer) — All Rows from Both

**Question:** Show EVERYONE — all employees and all salary records, NaN where there's no match.

```
employees              salaries
emp_id  name           emp_id  salary
  1     Alice    ←→      1     85000   ✓ matched
  2     Bob      ←→      2     72000   ✓ matched
  3     Carol    ←→      3     91000   ✓ matched
  4     Dave     ←→      (none)        ✓ KEPT, salary = NaN
  5     Eve      ←→      5     75000   ✓ matched
  (none) NaN     ←→      6     68000   ✓ KEPT, name = NaN
```

Result: 6 rows (all unique emp_ids from both tables)

In [21]:
# OUTER JOIN: keep ALL rows from both DataFrames, NaN where no match
outer_result = pd.merge(employees, salaries, on='emp_id', how='outer')

print("OUTER JOIN — all employees AND all salary records:")
display(outer_result)
print()
print("Notice: Dave has salary=NaN, emp_id=6 has name=NaN and dept_id=NaN")
print(f"Result shape: {outer_result.shape}  (expected 6 rows = union of both)")

OUTER JOIN — all employees AND all salary records:


,emp_id,name,dept_id,salary
0,1,Alice,10.0,85000.0
1,2,Bob,20.0,72000.0
2,3,Carol,10.0,91000.0
3,4,Dave,30.0,NaN
4,5,Eve,20.0,75000.0
5,6,NaN,NaN,68000.0



Notice: Dave has salary=NaN, emp_id=6 has name=NaN and dept_id=NaN
Result shape: (6, 4)  (expected 6 rows = union of both)


---
## 4. Merging on Multiple Columns

### When Do You Need This?

Sometimes a single column is **not enough** to uniquely identify a match. For example:

- A sales table has `(store_id, product_id)` as a composite key
- A budget table also has `(store_id, product_id)` — you need **both** columns to match

### Syntax

```python
pd.merge(left, right, on=['col1', 'col2'], how='inner')
```

Pandas will only match rows where **all** listed columns match simultaneously.

### Visual

```
sales_data             budget_data
store_id  product_id   store_id  product_id  budget
   A          101         A         101       5000   ← match (A, 101)
   A          102         A         102       3000   ← match (A, 102)
   B          101         B         101       4000   ← match (B, 101)
   B          103         B         999       2000   ← NO match
```

In [22]:
# Example: merging on multiple columns
# Imagine a store's sales vs. budget data — you need (store_id, product_id) to match

sales_data = pd.DataFrame({
    'store_id':   ['A', 'A', 'B', 'B'],
    'product_id': [101, 102, 101, 103],
    'units_sold': [50, 30, 45, 20],
})

budget_data = pd.DataFrame({
    'store_id':   ['A', 'A', 'B', 'B'],
    'product_id': [101, 102, 101, 999],
    'budget':     [5000, 3000, 4000, 2000],
})

print("sales_data:")
display(sales_data)
print()
print("budget_data:")
display(budget_data)

# Merge on BOTH store_id and product_id
multi_merge = pd.merge(sales_data, budget_data, on=['store_id', 'product_id'], how='inner')
print()
print("Inner merge on ['store_id', 'product_id']:")
display(multi_merge)
print()
print("Note: (B, 103) and (B, 999) don't match — dropped from inner join")

sales_data:


,store_id,product_id,units_sold
0,A,101,50
1,A,102,30
2,B,101,45
3,B,103,20



budget_data:


,store_id,product_id,budget
0,A,101,5000
1,A,102,3000
2,B,101,4000
3,B,999,2000



Inner merge on ['store_id', 'product_id']:


,store_id,product_id,units_sold,budget
0,A,101,50,5000
1,A,102,30,3000
2,B,101,45,4000



Note: (B, 103) and (B, 999) don't match — dropped from inner join


---
## 5. The `suffixes` Parameter — Handling Overlapping Column Names

### The Problem

When both DataFrames have a column with the **same name** (other than the key column), pandas can't tell them apart. It automatically appends suffixes: `_x` (left) and `_y` (right).

### Example

```python
# Both DataFrames have a column called 'bonus'
result = pd.merge(left_df, right_df, on='emp_id')
# Result columns: emp_id, bonus_x, bonus_y
```

### Fix: Use `suffixes=` to give meaningful names

```python
result = pd.merge(left_df, right_df, on='emp_id', suffixes=('_2023', '_2024'))
# Result columns: emp_id, bonus_2023, bonus_2024
```

### Common Mistake

> ❌ Ignoring `_x`/`_y` columns and using the wrong one without noticing
> ✅ Always check your column names after a merge with `result.columns`

In [23]:
# Demonstrating the _x / _y suffix problem and how to fix it

performance_2023 = pd.DataFrame({
    'emp_id': [1, 2, 3],
    'bonus':  [10000, 8000, 12000],   # bonus column!
})

performance_2024 = pd.DataFrame({
    'emp_id': [1, 2, 3],
    'bonus':  [11000, 9000, 13000],   # same column name!
})

display(performance_2023)
display(performance_2024)

# Without suffixes — pandas uses default _x and _y
default_merge = pd.merge(performance_2023, performance_2024, on='emp_id')
print("Without custom suffixes (confusing column names):")
display(default_merge)
print("Columns:", list(default_merge.columns))
print()

# With custom suffixes — much clearer!
clean_merge = pd.merge(
    performance_2023,
    performance_2024,
    on='emp_id',
    suffixes=('_2023', '_2024')
)
print("With custom suffixes (clear column names):")
display(clean_merge)
print("Columns:", list(clean_merge.columns))

,emp_id,bonus
0,1,10000
1,2,8000
2,3,12000


,emp_id,bonus
0,1,11000
1,2,9000
2,3,13000


Without custom suffixes (confusing column names):


,emp_id,bonus_x,bonus_y
0,1,10000,11000
1,2,8000,9000
2,3,12000,13000


Columns: ['emp_id', 'bonus_x', 'bonus_y']

With custom suffixes (clear column names):


,emp_id,bonus_2023,bonus_2024
0,1,10000,11000
1,2,8000,9000
2,3,12000,13000


Columns: ['emp_id', 'bonus_2023', 'bonus_2024']


---
## 6. Common Merge Bugs

Merging seems simple but has several silent failure modes. Let's look at each.

### Bug 1: Duplicate Rows (Many-to-Many Explosion)

If the key column has **duplicate values in both** DataFrames, the merge will produce a **Cartesian product** — every match from the left is paired with every match from the right.

```
left              right
id  value         id  category
 1  A              1  X
 1  B              1  Y

Inner join on id=1:
 1  A  X
 1  A  Y
 1  B  X
 1  B  Y   ← 4 rows from 2×2 = many-to-many!
```

### Bug 2: Wrong Join Type → Unexpected NaN

Forgetting `how='left'` when you need all rows from the left DataFrame will silently drop rows.

### Bug 3: Merging with Different dtypes

If one DataFrame has `emp_id` as **int** and another as **string**, they won't match:
```python
# id=1 (int) vs id='1' (string) → no match!
```
Fix: `df['emp_id'] = df['emp_id'].astype(int)` before merging.

### 🔑 Golden Rule
> **Always verify the shape of your result after every merge.**
> If it's larger or smaller than expected, something went wrong.

In [24]:
# Bug 1 Demo: Many-to-many explosion (duplicate keys in BOTH DataFrames)

orders = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'order_value': [200, 350, 100],
})

discounts = pd.DataFrame({
    'customer_id': [1, 1],      # customer 1 has TWO discount codes!
    'discount':    ['10%', '5%'],
})

print("orders (3 rows):")
display(orders)
print()
print("discounts (2 rows — customer_id=1 appears twice):")
display(discounts)
print()

many_to_many = pd.merge(orders, discounts, on='customer_id', how='inner')
print("Inner merge result:")
display(many_to_many)
print()
print("⚠️  Expected ~2 rows, got:", len(many_to_many))
print("Reason: customer_id=1 appeared 2x in orders and 2x in discounts → 2×2=4 rows!")

orders (3 rows):


,customer_id,order_value
0,1,200
1,1,350
2,2,100



discounts (2 rows — customer_id=1 appears twice):


,customer_id,discount
0,1,10%
1,1,5%



Inner merge result:


,customer_id,order_value,discount
0,1,200,10%
1,1,200,5%
2,1,350,10%
3,1,350,5%



⚠️  Expected ~2 rows, got: 4
Reason: customer_id=1 appeared 2x in orders and 2x in discounts → 2×2=4 rows!


In [25]:
# Bug 2 Demo: dtype mismatch — int vs string key

emp_int = pd.DataFrame({
    'emp_id': [1, 2, 3],         # integer
    'name':   ['Alice', 'Bob', 'Carol'],
})

sal_str = pd.DataFrame({
    'emp_id': ['1', '2', '3'],   # string!
    'salary': [85000, 72000, 91000],
})

print("emp_int dtypes:\n", emp_int.dtypes)
print()
print("sal_str dtypes:\n", sal_str.dtypes)
print()

# Merge with mismatched dtypes — likely zero rows!
bad_merge = pd.merge(emp_int, sal_str, on='emp_id', how='inner')
print("Merge result with dtype mismatch:")
print(bad_merge)
print(f"Shape: {bad_merge.shape}  ← 0 rows! No matches because int ≠ string")
print()

# Fix: convert before merging
sal_str['emp_id'] = sal_str['emp_id'].astype(int)
good_merge = pd.merge(emp_int, sal_str, on='emp_id', how='inner')
print("Fixed merge after converting emp_id to int:")
print(good_merge)

emp_int dtypes:
 emp_id     int64
name      object
dtype: object

sal_str dtypes:
 emp_id    object
salary     int64
dtype: object



ValueError: You are trying to merge on int64 and object columns for key 'emp_id'. If you wish to proceed you should use pd.concat

---
## 7. `df.join()` — The Index-Based Shortcut

### What Is It?

`df.join(other)` is a convenient method that joins two DataFrames **on their index** by default. Think of it as a shortcut for the special case where the key is the DataFrame index.

### Syntax

```python
left_df.join(right_df, how='left')        # join on index (default)
left_df.join(right_df, on='col_name')     # join left's column to right's index
```

### Return Value

A new DataFrame with columns from both DataFrames merged by index.

### When to Use `join` vs `merge`

| | `pd.merge()` | `df.join()` |
|--|--------------|-------------|
| **Key** | Any column(s) | Index (by default) |
| **Flexibility** | Very high | Lower |
| **Common use** | Most situations | When data is indexed by the key |
| **Overlapping columns** | Raises error or uses suffixes | Uses `lsuffix`/`rsuffix` |

> **Rule of thumb:** Use `pd.merge()` for almost everything. Use `df.join()` only when your DataFrames are already indexed by the join key.

In [26]:
# df.join() example — joining on DataFrame index

# Set emp_id as index in both DataFrames
emp_indexed = employees.set_index('emp_id')
sal_indexed = salaries.set_index('emp_id')

print("emp_indexed (index = emp_id):")
print(emp_indexed)
print()
print("sal_indexed (index = emp_id):")
print(sal_indexed)
print()

# Left join using df.join() — joins on the shared index
join_result = emp_indexed.join(sal_indexed, how='left')
print("join() result (left join on index):")
print(join_result)
print()
print("This is equivalent to:")
print("  pd.merge(employees, salaries, on='emp_id', how='left')")

emp_indexed (index = emp_id):
         name  dept_id
emp_id                
1       Alice       10
2         Bob       20
3       Carol       10
4        Dave       30
5         Eve       20

sal_indexed (index = emp_id):
        salary
emp_id        
1        85000
2        72000
3        91000
5        75000
6        68000

join() result (left join on index):
         name  dept_id   salary
emp_id                         
1       Alice       10  85000.0
2         Bob       20  72000.0
3       Carol       10  91000.0
4        Dave       30      NaN
5         Eve       20  75000.0

This is equivalent to:
  pd.merge(employees, salaries, on='emp_id', how='left')


---
## 🧪 Practice Questions

Work through these on your own before looking at the answers. The datasets `employees`, `departments`, and `salaries` from the setup cell are available.


In [29]:
lst = [employees,departments,salaries]
for data in lst:    
    display(data)

,emp_id,name,dept_id
0,1,Alice,10
1,2,Bob,20
2,3,Carol,10
3,4,Dave,30
4,5,Eve,20


,dept_id,dept_name,location
0,10,Engineering,Delhi
1,20,Marketing,Mumbai
2,30,HR,Pune
3,40,Finance,Bangalore


,emp_id,salary
0,1,85000
1,2,72000
2,3,91000
3,5,75000
4,6,68000


**Q1 [Easy]:** Inner merge `employees` with `departments` on `dept_id`. How many rows does the result have? What happened to dept_id=40 (Finance)?

In [30]:
pd.merge(employees,departments ,
         on='dept_id',
         how= 'inner')

,emp_id,name,dept_id,dept_name,location
0,1,Alice,10,Engineering,Delhi
1,2,Bob,20,Marketing,Mumbai
2,3,Carol,10,Engineering,Delhi
3,4,Dave,30,HR,Pune
4,5,Eve,20,Marketing,Mumbai


**Q2 [Easy]:** Left merge `employees` with `salaries` on `emp_id`. Who ends up with `salary = NaN`?

In [32]:
pd.merge(employees,salaries,
         on='emp_id',
         how= 'left')

,emp_id,name,dept_id,salary
0,1,Alice,10,85000.0
1,2,Bob,20,72000.0
2,3,Carol,10,91000.0
3,4,Dave,30,NaN
4,5,Eve,20,75000.0


**Q3 [Easy]:** Right merge `employees` with `salaries` on `emp_id`. Which row will have `name = NaN`?

In [33]:
pd.merge(employees,salaries,
         on='emp_id',
         how= 'right')

,emp_id,name,dept_id,salary
0,1,Alice,10.0,85000
1,2,Bob,20.0,72000
2,3,Carol,10.0,91000
3,5,Eve,20.0,75000
4,6,NaN,NaN,68000


**Q4 [Easy]:** Outer merge `employees` with `salaries` on `emp_id`. How many rows do you expect? Verify with `.shape`.

In [36]:
new_data = pd.merge(employees,salaries,
         on='emp_id',
         how= 'outer')
display(new_data)
new_data.shape

,emp_id,name,dept_id,salary
0,1,Alice,10.0,85000.0
1,2,Bob,20.0,72000.0
2,3,Carol,10.0,91000.0
3,4,Dave,30.0,NaN
4,5,Eve,20.0,75000.0
5,6,NaN,NaN,68000.0


(6, 4)

**Q5 [Medium]:** Chain two merges: first merge `employees` with `departments` on `dept_id`, then merge the result with `salaries` on `emp_id` (left join). Show a table with `name`, `dept_name`, `location`, and `salary`.

In [ ]:
df1 = pd.merge(employees,departments,
         on='dept_id',
         how= 'outer')

df2 = pd.merge(df1,salaries,
               on ='emp_id',
               how='left')

display(df2)

,emp_id,name,dept_id,dept_name,location,salary
0,1.0,Alice,10,Engineering,Delhi,85000.0
1,3.0,Carol,10,Engineering,Delhi,91000.0
2,2.0,Bob,20,Marketing,Mumbai,72000.0
3,5.0,Eve,20,Marketing,Mumbai,75000.0
4,4.0,Dave,30,HR,Pune,NaN
5,NaN,NaN,40,Finance,Bangalore,NaN


**Q6 [Medium]:** After left merging `employees` with `salaries`, find all employees who do **NOT** have a salary record. (Hint: filter rows where `salary` is NaN.)

In [42]:
import numpy as np
df = pd.merge(employees,salaries,
              on='emp_id',
              how='left')
display(df)
df[df['salary'].isna()]

,emp_id,name,dept_id,salary
0,1,Alice,10,85000.0
1,2,Bob,20,72000.0
2,3,Carol,10,91000.0
3,4,Dave,30,NaN
4,5,Eve,20,75000.0


,emp_id,name,dept_id,salary
3,4,Dave,30,NaN


**Q7 [Medium]:** Create a new DataFrame `emp_info` with columns `employee_number` (same values as `emp_id`) and `dept`. Merge it with `employees` using `left_on`/`right_on` since the column names differ.

In [ ]:
emp_info = pd.merge()

**Q8 [Medium]:** Create two DataFrames that both have an `emp_id` and a `score` column. Merge them, but use `suffixes=('_written', '_practical')` to clearly name the result columns.

---
## ✅ Answer Key

In [ ]:
# Q1 [Easy]: Inner merge employees with departments on dept_id

q1 = pd.merge(employees, departments, on='dept_id', how='inner')
print("Q1 — Inner merge (employees ∩ departments):")
print(q1)
print()
print(f"Shape: {q1.shape}")
print("Explanation: All 5 employees match a dept_id (10, 20, 30).")
print("Finance (dept_id=40) has NO employees, so it's dropped.")

In [ ]:
# Q2 [Easy]: Left merge employees with salaries

q2 = pd.merge(employees, salaries, on='emp_id', how='left')
print("Q2 — Left merge (all employees, salary if available):")
print(q2)
print()
missing_salary = q2[q2['salary'].isna()]
print("Employees with no salary record:")
print(missing_salary[['emp_id', 'name']])

In [ ]:
# Q3 [Easy]: Right merge employees with salaries

q3 = pd.merge(employees, salaries, on='emp_id', how='right')
print("Q3 — Right merge (all salary records, employee info if available):")
print(q3)
print()
unknown_emp = q3[q3['name'].isna()]
print("Salary records with no matching employee:")
print(unknown_emp)

In [ ]:
# Q4 [Easy]: Outer merge employees with salaries

q4 = pd.merge(employees, salaries, on='emp_id', how='outer')
print("Q4 — Outer merge (all employees + all salary records):")
print(q4)
print()
print(f"Shape: {q4.shape}")
print("Explanation: 5 employees + 1 unknown salary record (emp_id=6) = 6 unique emp_ids.")

In [ ]:
# Q5 [Medium]: Chain two merges — employees + departments + salaries

# Step 1: merge employees with departments
step1 = pd.merge(employees, departments, on='dept_id', how='inner')

# Step 2: merge that result with salaries (left join to keep all employees)
q5 = pd.merge(step1, salaries, on='emp_id', how='left')

print("Q5 — Chain merge (employees + departments + salaries):")
print(q5[['name', 'dept_name', 'location', 'salary']])

In [ ]:
# Q6 [Medium]: Find employees who do NOT have a salary record

emp_with_salary = pd.merge(employees, salaries, on='emp_id', how='left')

# Rows where salary is NaN → no matching record in salaries
no_salary = emp_with_salary[emp_with_salary['salary'].isna()]

print("Q6 — Employees with NO salary record:")
print(no_salary[['emp_id', 'name', 'dept_id', 'salary']])
print()
print("Explanation: Dave (emp_id=4) is in employees but NOT in salaries.")

In [ ]:
# Q7 [Medium]: Merge with different column names using left_on / right_on

# Create emp_info with 'employee_number' instead of 'emp_id'
emp_info = pd.DataFrame({
    'employee_number': [1, 2, 3, 4, 5],
    'dept':            [10, 20, 10, 30, 20],
})

print("emp_info (different column names):")
print(emp_info)
print()

# left_on refers to employees column, right_on refers to emp_info column
q7 = pd.merge(
    employees,
    emp_info,
    left_on='emp_id',
    right_on='employee_number',
    how='inner'
)

print("Q7 — Merge using left_on='emp_id', right_on='employee_number':")
print(q7[['emp_id', 'employee_number', 'name', 'dept_id', 'dept']])
print()
print("Note: Both 'emp_id' and 'employee_number' appear in the result.")
print("You can drop the redundant column with: q7.drop(columns='employee_number')")

In [ ]:
# Q8 [Medium]: Handle _x / _y suffix problem with custom suffixes

written_exam = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'score':  [88, 74, 92, 65],   # written exam scores
})

practical_exam = pd.DataFrame({
    'emp_id': [1, 2, 3, 4],
    'score':  [80, 70, 95, 60],   # practical exam scores
})

print("Without suffixes (confusing):")
bad = pd.merge(written_exam, practical_exam, on='emp_id')
print(bad)
print()

print("Q8 — With custom suffixes (clear):")
q8 = pd.merge(
    written_exam,
    practical_exam,
    on='emp_id',
    suffixes=('_written', '_practical')
)
print(q8)
print()

# Bonus: calculate total score
q8['total_score'] = q8['score_written'] + q8['score_practical']
print("With total_score:")
print(q8)

---
## 📋 Quick Revision Table

| Concept | Code | Key Point |
|---------|------|-----------|
| Basic merge | `pd.merge(left, right, on='col')` | Combines DataFrames on shared column |
| Inner join | `how='inner'` | Only rows matching in **both** (default) |
| Left join | `how='left'` | All left rows; NaN for missing right |
| Right join | `how='right'` | All right rows; NaN for missing left |
| Outer join | `how='outer'` | All rows from both; NaN where no match |
| Different key names | `left_on='a', right_on='b'` | Use when columns have different names |
| Multi-column key | `on=['col1', 'col2']` | All listed columns must match |
| Custom suffixes | `suffixes=('_x', '_y')` | Avoid confusing `_x`/`_y` defaults |
| Index-based join | `df.join(other)` | Joins on DataFrame index |
| Verify result | `result.shape` | Always check after merge |
| Find no-match rows | `result[result['col'].isna()]` | Filter NaN from join column |
| Fix dtype mismatch | `df['col'].astype(int)` | Both keys must have same dtype |

---
## 🎤 Interview Questions

**IQ1:** What is the difference between an inner join and a left join? When would you choose one over the other?

> **Answer:** An inner join returns **only rows that have a match in both DataFrames**. A left join returns **all rows from the left DataFrame**, with NaN for any right-side columns where no match exists. Use inner when you only care about matched records; use left when you must preserve all records from the primary table regardless of whether related data exists.

---

**IQ2:** After performing a merge, you notice that the result has far more rows than expected. What is the likely cause and how would you diagnose it?

> **Answer:** The likely cause is a **many-to-many relationship** — the key column has duplicate values in both DataFrames, causing a Cartesian product. To diagnose: (1) check `left['key'].duplicated().any()` and `right['key'].duplicated().any()`, (2) compare `result.shape` to what you expected, (3) inspect the duplicated rows with `result[result.duplicated(subset='key', keep=False)]`.

---

**IQ3:** What is the difference between `pd.merge()` and `df.join()`? When would you use each?

> **Answer:** `pd.merge()` is the general-purpose tool that joins on **any column(s)** and supports all join types with fine-grained control. `df.join()` is a convenience method that joins on the **DataFrame index** by default. Use `pd.merge()` in almost all cases; use `df.join()` only when your DataFrames are already indexed by the key you want to join on, as it is slightly more concise in that scenario.

---
## 🔭 What's Next?

**Notebook 9 — Concatenating Data**

Merging combines DataFrames **horizontally** (adding columns). Concatenation stacks DataFrames **vertically** (adding rows) or horizontally. In Notebook 9, you will learn:

- `pd.concat()` — stacking rows or columns
- `axis=0` vs `axis=1`
- Handling mismatched columns during concatenation
- `ignore_index=True` to reset the index
- When to use `concat` vs `merge`

---
*End of Notebook 8*